In [52]:
import crystal_toolkit
from pymatgen.core import Structure, Lattice
from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import get_symmetrically_distinct_miller_indices
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
import numpy as np

API_KEY = "2CMWZDlEGcEuPFYtfypcmSKwT5MddGRd"

with MPRester(API_KEY) as mpr:
    fe_bulk = mpr.get_structure_by_material_id("mp-13")
    zno_bulk = mpr.get_structure_by_material_id("mp-1986")

fe_bulk = SpacegroupAnalyzer(fe_bulk).get_conventional_standard_structure()
zno_bulk = SpacegroupAnalyzer(zno_bulk).get_conventional_standard_structure()

In [53]:
#Enumerate Miller Planes - limiting Miller index to 2.

max_index = 1

fe_planes = get_symmetrically_distinct_miller_indices(fe_bulk, max_index)
zno_planes = get_symmetrically_distinct_miller_indices(zno_bulk, max_index)

print("Fe planes:", fe_planes)
print("ZnO planes:", zno_planes)

Fe planes: [(1, 1, 1), (1, 1, 0), (1, 0, 0)]
ZnO planes: [(1, 1, 1), (1, 1, 0), (1, 0, 0)]


In [54]:
from pymatgen.analysis.interfaces.zsl import ZSLGenerator, ZSLMatch
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer, SubstrateMatch

zsl = ZSLGenerator(
    max_area_ratio_tol = 0.02,
    max_area = 400,
    max_length_tol = 0.02,
    max_angle_tol = 0.02
)

from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

cib_objects = []

for fe_plane in fe_planes:
    for zno_plane in zno_planes:
        
        try:
            cib = CoherentInterfaceBuilder(
                substrate_structure=fe_bulk,
                film_structure=zno_bulk,
                substrate_miller=fe_plane,
                film_miller=zno_plane,
                zslgen=zsl
            )

            cib_objects.append({
                "fe_plane": fe_plane,
                "zno_plane": zno_plane,
                "cib": cib
            })

        except:
            continue

print("Total CIB objects:", len(cib_objects))

Total CIB objects: 9


In [55]:
terminations = cib.terminations
print("terminations:", len(terminations))
for t in terminations:
    print(t)

terminations: 1
('O2_P4/mmm_1', 'Fe_P4/mmm_1')


In [56]:
termination_info = []

for item in cib_objects:
    
    cib = item["cib"]
    n_terminations = len(cib.terminations)

    termination_info.append({
        "fe_plane": item["fe_plane"],
        "zno_plane": item["zno_plane"],
        "terminations": n_terminations
    })

termination_info

[{'fe_plane': (1, 1, 1), 'zno_plane': (1, 1, 1), 'terminations': 2},
 {'fe_plane': (1, 1, 1), 'zno_plane': (1, 1, 0), 'terminations': 1},
 {'fe_plane': (1, 1, 1), 'zno_plane': (1, 0, 0), 'terminations': 1},
 {'fe_plane': (1, 1, 0), 'zno_plane': (1, 1, 1), 'terminations': 2},
 {'fe_plane': (1, 1, 0), 'zno_plane': (1, 1, 0), 'terminations': 1},
 {'fe_plane': (1, 1, 0), 'zno_plane': (1, 0, 0), 'terminations': 1},
 {'fe_plane': (1, 0, 0), 'zno_plane': (1, 1, 1), 'terminations': 2},
 {'fe_plane': (1, 0, 0), 'zno_plane': (1, 1, 0), 'terminations': 1},
 {'fe_plane': (1, 0, 0), 'zno_plane': (1, 0, 0), 'terminations': 1}]

In [57]:
matches = list(cib.zsl_matches)
print(len(matches))

859


In [58]:
match_info = []

for item in cib_objects:

    cib = item["cib"]

    matches = cib.zsl_matches

    match_info.append({
        "fe_plane": item["fe_plane"],
        "zno_plane": item["zno_plane"],
        "matches": len(matches)
    })

match_info

[{'fe_plane': (1, 1, 1), 'zno_plane': (1, 1, 1), 'matches': 213},
 {'fe_plane': (1, 1, 1), 'zno_plane': (1, 1, 0), 'matches': 38},
 {'fe_plane': (1, 1, 1), 'zno_plane': (1, 0, 0), 'matches': 144},
 {'fe_plane': (1, 1, 0), 'zno_plane': (1, 1, 1), 'matches': 1918},
 {'fe_plane': (1, 1, 0), 'zno_plane': (1, 1, 0), 'matches': 597},
 {'fe_plane': (1, 1, 0), 'zno_plane': (1, 0, 0), 'matches': 1266},
 {'fe_plane': (1, 0, 0), 'zno_plane': (1, 1, 1), 'matches': 672},
 {'fe_plane': (1, 0, 0), 'zno_plane': (1, 1, 0), 'matches': 220},
 {'fe_plane': (1, 0, 0), 'zno_plane': (1, 0, 0), 'matches': 859}]

In [59]:
all_interfaces = []

for item in cib_objects:
    cib = item["cib"]
    for term in cib.terminations:
        interfaces = list(
            cib.get_interfaces(
                termination=term,
                gap=1.75,
                film_thickness=2,
                substrate_thickness=4
            )
        )
        all_interfaces.extend(interfaces)

print("Total interfaces generated:", len(all_interfaces))

Total interfaces generated: 8730


In [60]:
interfaces_least_strain = min(all_interfaces,
           key=lambda x: x.interface_properties['von_mises_strain'])

print("Strain:", interfaces_least_strain.interface_properties['von_mises_strain']*100, "%")

Strain: 0.11529165572304932 %


In [62]:
import numpy as np

interfaces = []

for term in cib.terminations:
    ints = list(cib.get_interfaces(termination=term))
    interfaces.extend(ints)

for interface in interfaces:
    vecs = interface.interface_properties["substrate_sl_vectors"]

    a = np.array(vecs[0])
    b = np.array(vecs[1])

    area = np.linalg.norm(np.cross(a, b))

    interface.interface_properties["area"] = area


for interface in interfaces:
    strain = interface.interface_properties["von_mises_strain"]
    area   = interface.interface_properties["area"]

    print(f"strain = {strain:.6f}, area = {area:.2f}")

strain = 0.004613, area = 40.98
strain = 0.004613, area = 40.98
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.004613, area = 81.97
strain = 0.008536, area = 106.56
strain = 0.008536, area = 106.56
strain = 0.008536, area = 106.56
strain = 0.008536, area = 106.56
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, area = 122.95
strain = 0.004613, are

In [63]:
def score(interface):

    strain = interface.interface_properties["von_mises_strain"]
    area = interface.interface_properties["area"]

    return strain + area / 50


final_interface_ = min(all_interfaces, key=score)
final_interface_

KeyError: 'area'

In [30]:
final_interface_.to(filename="fe_zno_structure.cif")

"# generated using pymatgen\ndata_Zn2Fe5O2\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   6.40194200\n_cell_length_b   6.40194200\n_cell_length_c   39.74801673\n_cell_angle_alpha   90.00000000\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   Zn2Fe5O2\n_chemical_formula_sum   'Zn16 Fe40 O16'\n_cell_volume   1629.06695424\n_cell_formula_units_Z   8\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Zn  Zn0  1  0.00000000  0.50000000  0.63590257  1\n  Zn  Zn1  1  0.00000000  0.00000000  0.63590257  1\n  Zn  Zn2  1  0.50000000  0.50000000  0.63590257  1\n  Zn  Zn3  1  0.50000000  0.00000000  0.63590257  1\n  Zn  Zn4  1  0.25000000  0.25000000  0.57855996  1\n  Zn  Zn5  1  0.25000000  0.75000000  0

In [13]:
print("Fe transformation matrix:")
print(final_interface_.interface_properties["substrate_transformation"])

print("ZnO transformation matrix:")
print(final_interface_.interface_properties["film_transformation"])

Fe transformation matrix:
[[1. 2.]
 [0. 5.]]
ZnO transformation matrix:
[[2. 0.]
 [0. 2.]]


In [14]:
import numpy as np

f = final_interface_.interface_properties["substrate_sl_vectors"][0]
s = final_interface_.interface_properties["film_sl_vectors"][0]

angle = np.degrees(
        np.arccos(
        np.dot(f,s) / (np.linalg.norm(f)*np.linalg.norm(s))
    )
)

print("Rotation angle:", angle, 180 - angle)

Rotation angle: 161.565051177078 18.434948822921996


## Fe_Zno_Fe

In [64]:
from pymatgen.core import Structure
import numpy as np

struct = Structure.from_file("fe_zno_structure.cif")

coords = struct.cart_coords
z = coords[:, 2]

z_min = z.min()
z_max = z.max()

center_shift = -(z_min + z_max) / 2

struct.translate_sites(range(len(struct)), [0, 0, center_shift])

IOStream.flush timed out


In [5]:
# ── Cell B: Strip vacuum utility ─────────────────────────────────────────────
def strip_vacuum(s):
    """Set c = exact atomic thickness, shift z_min to 0."""
    z    = s.cart_coords[:, 2]
    zmin = z.min()
    zmax = z.max()
    new_coords = s.cart_coords.copy()
    new_coords[:, 2] -= zmin
    lat = s.lattice
    new_lat = Lattice.from_parameters(
        a=lat.a, b=lat.b, c=(zmax - zmin),
        alpha=lat.alpha, beta=lat.beta, gamma=lat.gamma
    )
    return Structure(new_lat, [site.specie for site in s],
                     new_coords, coords_are_cartesian=True)

bilayer = strip_vacuum(struct)
print(f"Bilayer c (atomic only) = {bilayer.lattice.c:.3f} Å")

Bilayer c (atomic only) = 47.115 Å


In [7]:
import numpy as np
from pymatgen.core import Structure, Lattice

# ── Step 1: Examine the bilayer z-distribution ───────────────────────────────
bilayer = strip_vacuum(Structure.from_file("fe_zno_structure.cif"))

# Print ALL sites sorted by z to understand the layout
print("All sites sorted by z:")

sites_sorted = sorted(bilayer, key=lambda s: s.coords[2])
for s in sites_sorted:
    print(f"  {s.specie.symbol:3s}  z = {s.coords[2]:.3f}")

All sites sorted by z:
  Fe   z = -0.000
  Fe   z = -0.000
  Fe   z = -0.000
  Fe   z = 0.000
  Fe   z = 0.000
  Fe   z = 1.432
  Fe   z = 1.432
  Fe   z = 1.432
  Fe   z = 1.432
  Fe   z = 1.432
  Fe   z = 2.863
  Fe   z = 2.863
  Fe   z = 2.863
  Fe   z = 2.863
  Fe   z = 2.863
  Fe   z = 4.295
  Fe   z = 4.295
  Fe   z = 4.295
  Fe   z = 4.295
  Fe   z = 4.295
  Fe   z = 5.726
  Fe   z = 5.726
  Fe   z = 5.726
  Fe   z = 5.726
  Fe   z = 5.726
  Fe   z = 7.158
  Fe   z = 7.158
  Fe   z = 7.158
  Fe   z = 7.158
  Fe   z = 7.158
  Fe   z = 8.589
  Fe   z = 8.589
  Fe   z = 8.589
  Fe   z = 8.589
  Fe   z = 8.589
  Fe   z = 10.021
  Fe   z = 10.021
  Fe   z = 10.021
  Fe   z = 10.021
  Fe   z = 10.021
  O    z = 11.771
  O    z = 11.771
  O    z = 11.771
  O    z = 11.771
  Zn   z = 12.910
  Zn   z = 12.910
  Zn   z = 12.910
  Zn   z = 12.910
  O    z = 14.050
  O    z = 14.050
  O    z = 14.050
  O    z = 14.050
  Zn   z = 15.190
  Zn   z = 15.190
  Zn   z = 15.190
  Zn   z = 15.190
 

In [8]:
# ── Step 2: Find the natural split point between Fe and ZnO ──────────────────
z_coords = np.array([s.coords[2] for s in bilayer])
z_sorted  = np.sort(z_coords)

# Find largest gap in z — that's the interface between Fe and ZnO
z_gaps   = np.diff(z_sorted)
split_idx = np.argmax(z_gaps)
z_split   = (z_sorted[split_idx] + z_sorted[split_idx + 1]) / 2

print(f"Split point z = {z_split:.3f} Å")
print(f"Gap at split  = {z_gaps[split_idx]:.3f} Å")

Split point z = 10.896 Å
Gap at split  = 1.750 Å


In [9]:
# ── Step 3: Split into two clean slabs at the interface ──────────────────────
def make_slab_from_sites(sites, ref_lattice):
    """Build a vacuum-free slab from a list of sites."""
    coords = np.array([s.coords for s in sites])
    z_min  = coords[:, 2].min()
    z_max  = coords[:, 2].max()
    
    new_coords = coords.copy()
    new_coords[:, 2] -= z_min          # shift so z starts at 0
    
    thickness = z_max - z_min
    # Add a small buffer (half typical bond length) so edge atoms aren't at c=0/c
    buffer = 1.5
    
    new_lat = Lattice.from_parameters(
        a=ref_lattice.a, b=ref_lattice.b, c=thickness + buffer,
        alpha=ref_lattice.alpha, beta=ref_lattice.beta, gamma=ref_lattice.gamma
    )
    return Structure(
        new_lat,
        [s.specie for s in sites],
        new_coords,
        coords_are_cartesian=True
    )

# Split: everything below z_split = slab_bottom, above = slab_top
slab_bottom_sites = [s for s in bilayer if s.coords[2] <= z_split]
slab_top_sites    = [s for s in bilayer if s.coords[2] >  z_split]

print(f"Bottom slab: {len(slab_bottom_sites)} sites → "
      f"{set(s.specie.symbol for s in slab_bottom_sites)}")
print(f"Top slab:    {len(slab_top_sites)} sites → "
      f"{set(s.specie.symbol for s in slab_top_sites)}")

slab_bottom = make_slab_from_sites(slab_bottom_sites, bilayer.lattice)
slab_top    = make_slab_from_sites(slab_top_sites,    bilayer.lattice)

Bottom slab: 40 sites → {'Fe'}
Top slab:    64 sites → {'O', 'Zn'}


In [10]:
# ── Step 4: Identify which is Fe and which is ZnO ────────────────────────────
def identify_slab(slab):
    symbols = set(s.specie.symbol for s in slab)
    return "Fe" if symbols == {"Fe"} else "ZnO" if symbols <= {"Zn","O"} else str(symbols)

print("Bottom slab is:", identify_slab(slab_bottom))
print("Top slab is:   ", identify_slab(slab_top))

# Assign correctly regardless of order
if identify_slab(slab_bottom) == "Fe":
    fe_slab, zno_slab = slab_bottom, slab_top
else:
    zno_slab, fe_slab = slab_bottom, slab_top

Bottom slab is: Fe
Top slab is:    ZnO


In [11]:
# ── Step 5: Stack  Fe | ZnO | Fe  with clean gaps ────────────────────────────
def stack_slabs(slabs, gaps, vacuum=15.0):
    ref = slabs[0].lattice
    current_z = vacuum
    z_offsets = []

    for i, slab in enumerate(slabs):
        z_offsets.append(current_z)
        current_z += slab.lattice.c
        if i < len(gaps):
            current_z += gaps[i]

    total_c = current_z + vacuum
    new_lat = Lattice.from_parameters(
        a=ref.a, b=ref.b, c=total_c,
        alpha=90, beta=90, gamma=90
    )

    all_species, all_coords = [], []
    for slab, z0 in zip(slabs, z_offsets):
        for site in slab:
            all_species.append(site.specie)
            all_coords.append([site.coords[0], site.coords[1],
                                site.coords[2] + z0])

    return Structure(new_lat, all_species, all_coords,
                     coords_are_cartesian=True)


fe_zno_fe = stack_slabs(
    slabs  = [fe_slab, zno_slab, fe_slab],
    gaps   = [0.5, 0.5],
    vacuum = 3.0
)
fe_zno_fe

In [12]:
fe_zno_fe.to(filename="fe_zno_fe_wrng.cif")

"# generated using pymatgen\ndata_Zn2Fe5O2\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   6.40194200\n_cell_length_b   6.40194200\n_cell_length_c   48.63566045\n_cell_angle_alpha   90.00000000\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   Zn2Fe5O2\n_chemical_formula_sum   'Zn32 Fe80 O32'\n_cell_volume   1993.32580130\n_cell_formula_units_Z   16\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Fe  Fe0  1  0.20000000  0.60000000  0.09111663  1\n  Fe  Fe1  1  0.60000000  0.80000000  0.09111663  1\n  Fe  Fe2  1  0.00000000  0.00000000  0.09111663  1\n  Fe  Fe3  1  0.40000000  0.20000000  0.09111663  1\n  Fe  Fe4  1  0.80000000  0.40000000  0.09111663  1\n  Fe  Fe5  1  0.50000000  0.50000000  

In [13]:
def flip_slab(slab):
    """
    Mirror slab in z so top becomes bottom and vice versa.
    This reverses which face is presented at the interface.
    """
    coords = slab.cart_coords.copy()
    c = slab.lattice.c
    
    # Mirror: z_new = c - z_old
    coords[:, 2] = c - coords[:, 2]
    
    # Shift so z_min = 0 again
    coords[:, 2] -= coords[:, 2].min()
    
    return Structure(
        slab.lattice,
        [s.specie for s in slab],
        coords,
        coords_are_cartesian=True
    )

fe_slab_flipped = flip_slab(fe_slab)

In [14]:
# Check what termination the flipped slab presents
fe_top_z    = sorted([s.coords[2] for s in fe_slab],         reverse=True)[:3]
fe_flip_z   = sorted([s.coords[2] for s in fe_slab_flipped], reverse=True)[:3]
zno_top_z   = sorted([s.coords[2] for s in zno_slab],        reverse=True)[:3]

print("Fe slab top 3 z:", [f"{z:.3f}" for z in fe_top_z])
print("Fe flipped top 3 z:", [f"{z:.3f}" for z in fe_flip_z])
print("ZnO top 3 z:", [f"{z:.3f}" for z in zno_top_z])

Fe slab top 3 z: ['10.021', '10.021', '10.021']
Fe flipped top 3 z: ['10.021', '10.021', '10.021']
ZnO top 3 z: ['17.094', '17.094', '17.094']


In [15]:
# Also check species at the top of each slab
def top_species(slab, n=4):
    sites = sorted(slab, key=lambda s: s.coords[2], reverse=True)[:n]
    return [s.specie.symbol for s in sites]

def bottom_species(slab, n=4):
    sites = sorted(slab, key=lambda s: s.coords[2])[:n]
    return [s.specie.symbol for s in sites]

print("\nZnO bottom (left interface):", bottom_species(zno_slab))
print("ZnO top    (right interface):", top_species(zno_slab))
print("\nFe slab bottom face:         ", bottom_species(fe_slab))
print("Fe flipped bottom face:      ", bottom_species(fe_slab_flipped))


ZnO bottom (left interface): ['O', 'O', 'O', 'O']
ZnO top    (right interface): ['Zn', 'Zn', 'Zn', 'Zn']

Fe slab bottom face:          ['Fe', 'Fe', 'Fe', 'Fe']
Fe flipped bottom face:       ['Fe', 'Fe', 'Fe', 'Fe']


In [16]:
# Rebuild with correct interface terminations:
# Left  interface: fe_slab top face   → zno_slab bottom face  (already correct)
# Right interface: zno_slab top face  → fe_slab_flipped bottom face
#
# Stack: fe_slab | zno_slab | fe_slab_flipped

fe_zno_fe_v2 = stack_slabs(
    slabs  = [fe_slab, zno_slab, fe_slab_flipped],
    gaps   = [2.0, 2.0],
    vacuum = 15.0
)

print(f"Fe|ZnO|Fe (correct terminations):")
print(f"  Total atoms : {len(fe_zno_fe_v2)}")
print(f"  Cell c      : {fe_zno_fe_v2.lattice.c:.3f} Å")

fe_zno_fe_v2

Fe|ZnO|Fe (correct terminations):
  Total atoms : 144
  Cell c      : 75.636 Å


## After fe_zno_structure.cif File

In [26]:
from pymatgen.core import Structure

# Load structure once
struct = Structure.from_file("fe_zno_structure.cif")

# Sort sites by z
coords_sorted = sorted(struct, key=lambda s: s.coords[2])

# Find two interface regions
all_z = [s.coords[2] for s in coords_sorted]
gaps = np.diff(all_z)
gap_indices = np.argsort(gaps)[-2:]  # two largest gaps

for gi in sorted(gap_indices):
    z_interface = (all_z[gi] + all_z[gi + 1]) / 2

    below = [s for s in struct if z_interface - 3 < s.coords[2] < z_interface]
    above = [s for s in struct if z_interface < s.coords[2] < z_interface + 3]

    print(f"\nInterface at z={z_interface:.2f} Å:")
    print(f"  Below: {[s.specie.symbol for s in sorted(below, key=lambda s: s.coords[2])]}")
    print(f"  Above: {[s.specie.symbol for s in sorted(above, key=lambda s: s.coords[2])]}")


Interface at z=19.31 Å:
  Below: ['Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe']
  Above: ['Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'O', 'O', 'O', 'O']

Interface at z=20.90 Å:
  Below: ['Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe', 'Fe']
  Above: ['O', 'O', 'O', 'O', 'Zn', 'Zn', 'Zn', 'Zn']


In [27]:
# Step 1: Check ALL terminations available in your original cib
print("All available terminations:")
for i, t in enumerate(cib.terminations):
    print(f"  [{i}] {t}")
    # termination is a tuple: (film_termination, substrate_termination)
    # film = ZnO surface, substrate = Fe surface

# Fixed score function - computes area inline, no pre-computation needed
def score(iface):
    strain = iface.interface_properties['von_mises_strain']
    vecs   = iface.interface_properties['substrate_sl_vectors']
    area   = np.linalg.norm(np.cross(vecs[0], vecs[1]))
    return strain + area / 50

All available terminations:
  [0] ('O2_P4/mmm_1', 'Fe_P4/mmm_1')


In [28]:
# Now re-run Step 2 with the fixed score function
for term in cib.terminations:
    ifaces = list(cib.get_interfaces(
        termination=term,
        gap=1.75,
        film_thickness=4,
        substrate_thickness=4
    ))
    if not ifaces:
        continue
    
    best         = min(ifaces, key=score)
    s            = strip_vacuum(Structure.from_sites(best))
    sites_sorted = sorted(s, key=lambda x: x.coords[2])
    
    print(f"\nTermination: {term}")
    print(f"  Bottom 4 (Fe side): {[x.specie.symbol for x in sites_sorted[:4]]}")
    print(f"  Top 4    (ZnO side): {[x.specie.symbol for x in sites_sorted[-4:]]}")


Termination: ('O2_P4/mmm_1', 'Fe_P4/mmm_1')
  Bottom 4 (Fe side): ['Fe', 'Fe', 'Fe', 'Fe']
  Top 4    (ZnO side): ['Zn', 'Zn', 'Zn', 'Zn']


In [29]:
# Step 3: Once you identify the O-terminated ZnO termination,
# extract slabs from that SINGLE bilayer

# From your earlier output the termination was:
# ('O2_P4/mmm_1', 'Fe_P4/mmm_1')
# 'O2_P4/mmm_1' means ZnO is O-terminated on the Fe-facing side ← this is what we want

# Get the best interface for this specific termination
target_term = ('O2_P4/mmm_1', 'Fe_P4/mmm_1')  # adjust if different

ifaces = list(cib.get_interfaces(
    termination=target_term,
    gap=1.75,
    film_thickness=4,
    substrate_thickness=4
))
best   = min(ifaces, key=score)
bilayer = strip_vacuum(Structure.from_sites(best))

print(bilayer, "Chosen bilayer")

Full Formula (Zn32 Fe40 O32)
Reduced Formula: Zn4Fe5O4
abc   :   6.401942   6.401942  28.865037
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (104)
  #  SP       a      b          c
---  ----  ----  -----  ---------
  0  Zn    0      0.5    0.526225
  1  Zn    0      0      0.526225
  2  Zn    0.5    0.5    0.526225
  3  Zn    0.5    0      0.526225
  4  Zn    0.25   0.25   0.447263
  5  Zn    0.25   0.75   0.447263
  6  Zn    0.75   0.25   0.447263
  7  Zn    0.75   0.75   0.447263
  8  Zn    0      0.5    0.68415
  9  Zn    0      0      0.68415
 10  Zn    0.5    0.5    0.68415
 11  Zn    0.5    0      0.68415
 12  Zn    0.25   0.25   0.605188
 13  Zn    0.25   0.75   0.605188
 14  Zn    0.75   0.25   0.605188
 15  Zn    0.75   0.75   0.605188
 16  Zn    0      0.5    0.842075
 17  Zn    0      1      0.842075
 18  Zn    0.5    0.5    0.842075
 19  Zn    0.5    0      0.842075
 20  Zn    0.25   0.25   0.763113
 21  Zn    0.25   0.75   0.76311

In [30]:
# Step 4: Split into Fe slab and ZnO slab
def split_bilayer(bl):
    z = np.array([s.coords[2] for s in bl])
    zs = np.sort(z)
    sp = (zs[np.argmax(np.diff(zs))] + zs[np.argmax(np.diff(zs))+1]) / 2
    def mk(sites):
        c = np.array([s.coords for s in sites]); c[:,2] -= c[:,2].min()
        l = bl.lattice
        return Structure(Lattice.from_parameters(l.a,l.b,max(c[:,2].max(),1),l.alpha,l.beta,l.gamma),
                         [s.specie for s in sites], c, coords_are_cartesian=True)
    return mk([s for s in bl if s.coords[2]<=sp]), mk([s for s in bl if s.coords[2]>sp])

def flip_slab(s):
    c = s.cart_coords.copy(); c[:,2] = s.lattice.c - c[:,2]; c[:,2] -= c[:,2].min()
    return Structure(s.lattice, [x.specie for x in s], c, coords_are_cartesian=True)

def stack_slabs(slabs, gaps, vacuum=15.0):
    z0, offsets = vacuum, []
    for i,s in enumerate(slabs):
        offsets.append(z0); z0 += s.lattice.c + (gaps[i] if i<len(gaps) else 0)
    l = slabs[0].lattice
    new_l = Lattice.from_parameters(l.a,l.b,z0+vacuum,90,90,90)
    sp,co = [],[]
    for s,z in zip(slabs,offsets):
        for site in s: sp.append(site.specie); co.append(site.coords+[0,0,z])
    return Structure(new_l, sp, co, coords_are_cartesian=True)

def faces(s,n=4):
    t = sorted(s, key=lambda x: x.coords[2])
    print(f"  bot:{[x.specie.symbol for x in t[:n]]}  top:{[x.specie.symbol for x in t[-n:]]}")

fe_slab, zno_slab = split_bilayer(bilayer)
zno_sites = sorted(zno_slab, key=lambda s: s.coords[2])
print("ZnO bottom face:", [s.specie.symbol for s in zno_sites[:4]])
print("ZnO top face:   ", [s.specie.symbol for s in zno_sites[-4:]])

ZnO bottom face: ['O', 'O', 'O', 'O']
ZnO top face:    ['Zn', 'Zn', 'Zn', 'Zn']


In [31]:
# Step 6: Find the ZnO thickness that gives O termination on BOTH faces
print("Searching for O-O terminated ZnO slab...\n")

for thickness in [2, 3, 4, 5, 6, 7, 8]:
    ifaces = list(cib.get_interfaces(
        termination=target_term,
        gap=1.75,
        film_thickness=thickness,   # vary this
        substrate_thickness=4
    ))
    if not ifaces:
        continue
    
    best    = min(ifaces, key=score)
    bilayer = strip_vacuum(Structure.from_sites(best))
    _, zno  = split_bilayer(bilayer)
    
    zno_sorted = sorted(zno, key=lambda s: s.coords[2])
    bot = zno_sorted[0].specie.symbol
    top = zno_sorted[-1].specie.symbol
    
    print(f"  film_thickness={thickness}: "
          f"ZnO bottom={bot}, top={top}  "
          f"{'← O-O termination!' if bot=='O' and top=='O' else ''}")

Searching for O-O terminated ZnO slab...

  film_thickness=2: ZnO bottom=O, top=Zn  
  film_thickness=3: ZnO bottom=O, top=Zn  
  film_thickness=4: ZnO bottom=O, top=Zn  
  film_thickness=5: ZnO bottom=O, top=Zn  
  film_thickness=6: ZnO bottom=O, top=Zn  
  film_thickness=7: ZnO bottom=O, top=Zn  
  film_thickness=8: ZnO bottom=O, top=Zn  


In [33]:
# Search O-O ZnO termination
term = ('O2_P4/mmm_1','Fe_P4/mmm_1')
for t in range(2,12):
    ifaces = list(cib.get_interfaces(termination=term,gap=1.75,film_thickness=t,substrate_thickness=4))
    if not ifaces: continue
    _, zno = split_bilayer(strip_vacuum(Structure.from_sites(min(ifaces,key=score))))
    z = sorted(zno, key=lambda s: s.coords[2])
    oo = "← O-O!" if z[0].specie.symbol==z[-1].specie.symbol=='O' else ""
    print(f"t={t}: bot={z[0].specie.symbol} top={z[-1].specie.symbol} {oo}")

t=2: bot=O top=Zn 
t=3: bot=O top=Zn 
t=4: bot=O top=Zn 
t=5: bot=O top=Zn 
t=6: bot=O top=Zn 
t=7: bot=O top=Zn 
t=8: bot=O top=Zn 
t=9: bot=O top=Zn 
t=10: bot=O top=Zn 
t=11: bot=O top=Zn 


In [33]:
# Build final structure — set oo_t to whichever thickness printed "← O-O!"
oo_t = 4  # <-- change this

ifaces_fe  = list(cib.get_interfaces(termination=term,gap=1.75,film_thickness=4,substrate_thickness=4))
ifaces_zno = list(cib.get_interfaces(termination=term,gap=1.75,film_thickness=oo_t,substrate_thickness=4))

fe_slab, _   = split_bilayer(strip_vacuum(Structure.from_sites(min(ifaces_fe, key=score))))
_, zno_slab  = split_bilayer(strip_vacuum(Structure.from_sites(min(ifaces_zno,key=score))))

faces(fe_slab,  4); faces(zno_slab, 4)

result = stack_slabs([fe_slab, zno_slab, flip_slab(fe_slab)], [2.0,2.0])
result

  bot:['Fe', 'Fe', 'Fe', 'Fe']  top:['Fe', 'Fe', 'Fe', 'Fe']
  bot:['O', 'O', 'O', 'O']  top:['Zn', 'Zn', 'Zn', 'Zn']


## method 2

In [2]:
from pymatgen.core import Structure

struct = Structure.from_file("fe_zno_structure.cif")
struct

In [28]:
# ONE CELL - complete clean rewrite
import numpy as np
from pymatgen.core import Structure, Lattice

def strip_vacuum(s):
    z = s.cart_coords[:,2]
    c = s.cart_coords.copy(); c[:,2] -= z.min()
    l = s.lattice
    return Structure(
        Lattice.from_parameters(l.a,l.b,z.max()-z.min(),l.alpha,l.beta,l.gamma),
        [x.specie for x in s], c, coords_are_cartesian=True)

def score(i):
    v = i.interface_properties['substrate_sl_vectors']
    return i.interface_properties['von_mises_strain'] + \
           np.linalg.norm(np.cross(v[0],v[1]))/50

def split_bilayer(bl):
    z  = np.sort([s.coords[2] for s in bl])
    sp = (z[np.argmax(np.diff(z))] + z[np.argmax(np.diff(z))+1]) / 2
    def mk(sites):
        c = np.array([s.coords for s in sites]); c[:,2] -= c[:,2].min()
        l = bl.lattice
        return Structure(
            Lattice.from_parameters(l.a,l.b,max(c[:,2].max(),1),l.alpha,l.beta,l.gamma),
            [s.specie for s in sites], c, coords_are_cartesian=True)
    return mk([s for s in bl if s.coords[2]<=sp]), mk([s for s in bl if s.coords[2]>sp])

def flip_slab(s):
    c = s.cart_coords.copy()
    c[:,2] = s.lattice.c - c[:,2]; c[:,2] -= c[:,2].min()
    return Structure(s.lattice,[x.specie for x in s],c,coords_are_cartesian=True)

def stack_slabs(slabs, gaps, vacuum=15.0):
    z0, offsets = vacuum, []
    for i,s in enumerate(slabs):
        offsets.append(z0); z0 += s.lattice.c+(gaps[i] if i<len(gaps) else 0)
    l = slabs[0].lattice
    nl = Lattice.from_parameters(l.a,l.b,z0+vacuum,90,90,90)
    sp,co = [],[]
    for s,z in zip(slabs,offsets):
        for site in s:
            sp.append(site.specie); co.append(site.coords+np.array([0,0,z]))
    return Structure(nl,sp,co,coords_are_cartesian=True)

def faces(s, name=""):
    t = sorted(s, key=lambda x: x.coords[2])
    print(f"{name} | bot:{[x.specie.symbol for x in t[:4]]} top:{[x.specie.symbol for x in t[-4:]]}")


In [8]:
# Get bilayer with O on the CIB interface side (bot=O confirmed)
term = ('O2_P4/mmm_1','Fe_P4/mmm_1')
ifaces = list(cib.get_interfaces(termination=term,gap=1.75,
              film_thickness=2,substrate_thickness=4))
bilayer = strip_vacuum(Structure.from_sites(min(ifaces,key=score)))

fe_slab, zno_slab = split_bilayer(bilayer)
faces(zno_slab, "ZnO")  # expect bot=O, top=Zn

ZnO | bot:['O', 'O', 'O', 'O'] top:['Zn', 'Zn', 'Zn', 'Zn']


In [9]:
# Flip the entire bilayer → ZnO(bot=Zn,top=O) | Fe
# This gives us the RIGHT interface with O-Fe bonding
bilayer_flipped = flip_slab(bilayer)
zno_flipped, fe_flipped = split_bilayer(bilayer_flipped)

faces(zno_slab,    "ZnO original : ")  # bot=O, top=Zn
faces(zno_flipped, "ZnO flipped  : ")  # bot=Zn, top=O

ZnO original :  | bot:['O', 'O', 'O', 'O'] top:['Zn', 'Zn', 'Zn', 'Zn']
ZnO flipped  :  | bot:['Zn', 'Zn', 'Zn', 'Zn'] top:['O', 'O', 'O', 'O']


In [10]:
# Stack: fe_slab | zno_slab | zno_flipped | fe_flipped
#
# Left  interface:  Fe(top=Fe) → ZnO(bot=O)   = Fe-O bond ✓
# Middle junction:  ZnO(top=Zn) → ZnO(bot=Zn) = Zn-Zn (same material, small gap)
# Right interface:  ZnO(top=O)  → Fe(bot=Fe)   = O-Fe bond ✓

result = stack(
    [fe_slab, zno_slab, zno_flipped, fe_flipped],
    gaps=[2.0, 1.0, 2.0]  # smaller gap at Zn-Zn junction
)

faces(result, "FINAL")
print(f"atoms={len(result)} c={result.lattice.c:.2f}A")
result

FINAL | bot:['Fe', 'Fe', 'Fe', 'Fe'] top:['Fe', 'Fe', 'Fe', 'Fe']
atoms=144 c=71.00A


/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/pymatgen/analysis/local_env.py:4021: UserWarning:

No oxidation states specified on sites! For better results, set the site oxidation states in the structure.

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/pymatgen/analysis/local_env.py:4023: UserWarning:

No oxidation states specified on sites! For better results, set the site oxidation states in the structure.

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/pymatgen/analysis/local_env.py:3935: UserWarning:

CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.



In [11]:
# Remove the TOP Zn layer of zno_slab (or bottom of zno_flipped — same thing)
def remove_top_layer(slab):
    """Remove the topmost atomic layer."""
    sites = sorted(slab, key=lambda s: s.coords[2])
    top_z = sites[-1].coords[2]
    # find all sites in the top layer (within 0.5 Å)
    keep  = [s for s in slab if abs(s.coords[2] - top_z) > 0.5]
    coords = np.array([s.coords for s in keep])
    coords[:,2] -= coords[:,2].min()
    l = slab.lattice
    return Structure(
        Lattice.from_parameters(l.a,l.b,coords[:,2].max(),l.alpha,l.beta,l.gamma),
        [s.specie for s in keep], coords, coords_are_cartesian=True)

zno_left = remove_top_layer(zno_slab)   # removes top Zn → now bot=O, top=O
faces(zno_left, "ZnO left trimmed")     # should be bot=O, top=O ✓

ZnO left trimmed | bot:['O', 'O', 'O', 'O'] top:['O', 'O', 'O', 'O']


In [16]:
# Now stack: fe | zno_left(O-O) | zno_flipped | fe_flipped
# Junction: zno_left(top=O) → zno_flipped(bot=Zn) = O-Zn ✓

result_2 = stack(
    [fe_slab, zno_left, zno_flipped, fe_flipped],
    gaps=[2.0, 0.0, 2.0]  # gap=0 at O-Zn junction, let it bond naturally
)

faces(result_2, "FINAL")
print(f"atoms={len(result)}  c={result.lattice.c:.2f}A")
result_2

FINAL | bot:['Fe', 'Fe', 'Fe', 'Fe'] top:['Fe', 'Fe', 'Fe', 'Fe']
atoms=140  c=98.86A


In [38]:
def strip_top_zn_layer(s):
    import numpy as np
    coords = np.array([site.coords for site in s])
    z_vals = np.round(coords[:, 2], 1)
    unique_z = np.sort(np.unique(z_vals))
    top_z = unique_z[-1]  # exactly 8.0
    
    keep = [i for i, site in enumerate(s) if np.round(site.coords[2], 1) != top_z]
    
    kept_sites = [s[i] for i in keep]
    kept_coords = coords[keep].copy()
    kept_coords[:, 2] -= kept_coords[:, 2].min()
    
    l = s.lattice
    new_lattice = Lattice.from_parameters(l.a, l.b, kept_coords[:, 2].max() + 1,
                                          l.alpha, l.beta, l.gamma)
    return Structure(new_lattice, [site.specie for site in kept_sites],
                     kept_coords, coords_are_cartesian=True)

In [39]:
zno_slab_trimmed = strip_top_zn_layer(zno_slab)

# Verify: should now end at z=6.8 → O on top
coords2 = np.array([site.coords for site in zno_slab_trimmed])
z2 = np.round(coords2[:, 2], 1)
for z in np.sort(np.unique(z2)):
    mask = z2 == z
    print(f"  z={z:.1f} → {[site.specie.symbol for site in zno_slab_trimmed][np.where(mask)[0][0]]}")

  z=0.0 → O
  z=1.1 → Zn
  z=2.3 → O
  z=3.4 → Zn
  z=4.6 → O
  z=5.7 → Zn
  z=6.8 → O


In [45]:
resultant = stack_slabs(
    [fe_slab, zno_slab_trimmed, fe_flipped],
    [1.75, 0.75]   # smaller left gap to match bond length
)
faces(resultant, "FINAL")
resultant

FINAL | bot:['Fe', 'Fe', 'Fe', 'Fe'] top:['Fe', 'Fe', 'Fe', 'Fe']


In [46]:
resultant.to(filename="fe_zno_fe_final1.cif")

"# generated using pymatgen\ndata_Zn3(Fe5O)4\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   6.40194200\n_cell_length_b   6.40194200\n_cell_length_c   60.37901348\n_cell_angle_alpha   90.00000000\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   Zn3(Fe5O)4\n_chemical_formula_sum   'Zn12 Fe80 O16'\n_cell_volume   2474.62549532\n_cell_formula_units_Z   4\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Fe  Fe0  1  0.20000000  0.60000000  0.27213955  1\n  Fe  Fe1  1  0.60000000  0.80000000  0.27213955  1\n  Fe  Fe2  1  0.00000000  -0.00000000  0.27213955  1\n  Fe  Fe3  1  0.40000000  0.20000000  0.27213955  1\n  Fe  Fe4  1  0.80000000  0.40000000  0.27213955  1\n  Fe  Fe5  1  0.50000000  0.500000

In [ ]:
from pymatgen.core import Structure
import crystal_toolkit

fe_zno_fe = Structure.from_file("fe_zno_fe_final1.cif")
fe_zno_fe